In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from neo4j import GraphDatabase
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import os

warnings.filterwarnings('ignore')

# --- Neo4j Connection Details ---
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "ubuntuubuntu" # Use the password you set on neo4j-container-maker.sh

# ==============================================================================
# === STEP 1: REAL DATA LOADING AND PREPARATION (REPLACES LOG GENERATION) ======
# ==============================================================================

def get_directory_urls(base_url):
    """Gets all directory URLs starting with '2024' from the base URL."""
    try:
        response = requests.get(base_url)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        return [urljoin(base_url, link.get('href')) for link in soup.find_all('a') if link.get('href') and link.get('href').startswith('2024') and link.get('href').endswith('/')]
    except requests.exceptions.RequestException as e:
        print(f"Error fetching base URL: {e}")
        return []

def load_real_zeek_data():
    """Loads and combines all parquet files from the UWF dataset."""
    BASE_URL = "https://datasets.uwf.edu/data/UWF-ZeekData24/parquet/"
    all_dataframes = []
    print("Starting data collection from UWF dataset...")
    directory_urls = get_directory_urls(BASE_URL)
    
    for dir_url in directory_urls:
        print(f"  Processing directory: {dir_url.strip('/').split('/')[-1]}")
        try:
            response = requests.get(dir_url)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            parquet_files = [urljoin(dir_url, link.get('href')) for link in soup.find_all('a') if link.get('href') and link.get('href').endswith('.parquet')]
            for file_url in parquet_files:
                df = pd.read_parquet(file_url, engine='pyarrow')
                all_dataframes.append(df)
        except Exception as e:
            print(f"    Could not process {dir_url}: {e}")

    if not all_dataframes:
        print("No dataframes were loaded. Exiting.")
        return None

    print("\nCombining and cleaning data...")
    combined_df = pd.concat(all_dataframes, ignore_index=True)
    
    # Clean the data by removing noisy 'Duplicate' labels
    cleaned_df = combined_df[combined_df['label_technique'] != 'Duplicate'].copy()
    cleaned_df['datetime'] = pd.to_datetime(cleaned_df['datetime'])
    print(f"Data loading complete. Total cleaned rows: {len(cleaned_df)}")
    return cleaned_df

combined_df = load_real_zeek_data()

Starting data collection from UWF dataset...
  Processing directory: 2024-02-25%20-%202024-03-03
  Processing directory: 2024-03-03%20-%202024-03-10
  Processing directory: 2024-03-10%20-%202024-03-17
  Processing directory: 2024-03-17%20-%202024-03-24
  Processing directory: 2024-03-24%20-%202024-03-31
  Processing directory: 2024-10-27%20-%202024-11-03
  Processing directory: 2024-11-03%20-%202024-11-10

Combining and cleaning data...
Data loading complete. Total cleaned rows: 1898613


In [17]:
combined_df.columns

Index(['community_id', 'conn_state', 'duration', 'history', 'src_ip_zeek',
       'src_port_zeek', 'dest_ip_zeek', 'dest_port_zeek', 'local_orig',
       'local_resp', 'missed_bytes', 'orig_bytes', 'orig_ip_bytes',
       'orig_pkts', 'proto', 'resp_bytes', 'resp_ip_bytes', 'resp_pkts',
       'service', 'ts', 'uid', 'datetime', 'label_tactic', 'label_technique',
       'label_binary', 'label_cve'],
      dtype='object')

In [16]:
combined_df.label_tactic.value_counts()

label_tactic
none                 958109
Credential Access    871188
Reconnaissance        58095
Defense Evasion        6048
Initial Access         4614
Exfiltration            559
Name: count, dtype: int64

In [ ]:


# ==============================================================================
# === STEP 2: ADAPT TTP MAPPING TO USE REAL DATA LABELS ========================
# ==============================================================================

def map_to_attack_ttp(row):
    """Maps attack properties based on the real labels in the dataframe."""
    props = {"is_attack": False, "early_stage": False, "late_stage": False, "ttp_tag": "Benign", "tactic": "Benign"}
    
    # Use the 'label_binary' column to identify any malicious activity
    if row['label_binary']:
        props["is_attack"] = True
        props["ttp_tag"] = row['label_technique']
        props["tactic"] = row['label_tactic']
        
        # Classify tactics into early or late stage for the hypothesis test
        early_stage_tactics = ['Reconnaissance', 'Resource Development', 'Initial Access']
        if props["tactic"] in early_stage_tactics:
            props["early_stage"] = True
        else:
            props["late_stage"] = True # All other malicious tactics are considered late-stage
            
    return props

# ==============================================================================
# === STEP 3: UPDATE GRAPH BUILDING TO USE THE NEW DATAFRAME ===================
# ==============================================================================

def build_graph_from_dataframe(driver, df):
    """Builds the Neo4j graph from the provided pandas DataFrame."""
    print(f"\nBuilding graph from {len(df)} log entries...")
    
    with driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")
        print("Database cleared. Starting fresh graph build.")
        
        # Get all unique IP addresses from source and destination columns
        all_ips = pd.concat([df['src_ip_zeek'], df['dest_ip_zeek']]).unique().tolist()
        print(f"Creating {len(all_ips)} unique IP nodes...")
        node_query = "UNWIND $ip_list AS ip MERGE (n:IP {address: ip})"
        session.run(node_query, ip_list=all_ips)

        print("Creating relationships and applying ATT&CK tags...")
        # Prepare data for batch upload
        relationship_data = []
        for index, row in df.iterrows():
            record = row.to_dict()
            record.update(map_to_attack_ttp(row))
            relationship_data.append(record)
        
        # Cypher query to create relationships from the data
        relationship_query = """
        UNWIND $rows AS row
        MATCH (orig:IP {address: row.src_ip_zeek}), (resp:IP {address: row.dest_ip_zeek})
        CREATE (orig)-[:CONNECTS {
            timestamp: row.ts, duration: row.duration, service: row.service,
            port: row.dest_port_zeek, state: row.conn_state, tactic: row.tactic,
            ttp_tag: row.ttp_tag, is_attack: row.is_attack, 
            is_attack_early: row.early_stage, is_attack_late: row.late_stage
        }]->(resp)
        """
        session.run(relationship_query, rows=relationship_data)
        print("Neo4j graph construction complete.")

# ==============================================================================
# === STEP 4: HYPOTHESIS TEST & VERIFICATION (LARGELY UNCHANGED) ===============
# ==============================================================================

def test_thesis_hypothesis(driver):
    """Runs the GDS analysis to test the hypothesis."""
    print("\n--- Starting Thesis Hypothesis Test ---")
    with driver.session() as session:
        session.run("CALL gds.graph.drop('thesisGraph', false)")
        print("1. Projecting graph into GDS...")
        session.run("CALL gds.graph.project('thesisGraph', 'IP', 'CONNECTS')")
        print("2. Generating node embeddings with FastRP...")
        session.run("CALL gds.fastRP.mutate('thesisGraph', {embeddingDimension: 128, mutateProperty: 'embedding'})")
        print("3. Fetching embeddings for correlation analysis...")
        
        # This query now finds ANY node involved in early/late attacks, not a specific one
        early_result = session.run("MATCH (a:IP)-[r:CONNECTS {is_attack_early: true}]->() RETURN gds.util.asNode(id(a)).embedding AS embedding LIMIT 1").single()
        late_result = session.run("MATCH (a:IP)-[r:CONNECTS {is_attack_late: true}]->() RETURN gds.util.asNode(id(a)).embedding AS embedding LIMIT 1").single()
        
        if not early_result or not late_result:
            print("Could not find embeddings for both early and late-stage attacks. Aborting test.")
            return

        early_embedding = np.array(early_result['embedding']).reshape(1, -1)
        late_embedding = np.array(late_result['embedding']).reshape(1, -1)
        similarity_score = cosine_similarity(early_embedding, late_embedding)[0][0]
        
        print("\n--- HYPOTHESIS TEST RESULTS ---")
        print(f"Cosine Similarity between an Early and a Late-Stage Attacker Node: {similarity_score:.4f}")
        if similarity_score > 0.85:
            print("Result: High correlation found. The hypothesis is strongly supported by this real-world data.")
        else:
            print("Result: Low correlation found. The hypothesis is not supported by this real-world data.")
        session.run("CALL gds.graph.drop('thesisGraph', false)")

def run_verification_query(driver):
    """Finds and prints attack paths in the graph."""
    print("\n--- Automated Verification: Finding Attack Path ---")
    query = """
    MATCH (a:IP)-[r:CONNECTS {is_attack_late: TRUE}]->(v:IP) 
    RETURN a.address AS attacker_ip, v.address AS victim_ip, r.port AS port, r.tactic AS tactic, r.ttp_tag AS technique
    LIMIT 10
    """
    with driver.session() as session:
        result = session.run(query).data()
        if result:
            for record in result:
                print(f"Attack Detected: {record['attacker_ip']} -> {record['victim_ip']} on port {record['port']} ({record['tactic']}/{record['technique']})")
        else:
            print("Verification Failed: No late-stage attack path found in the graph.")

# ==============================================================================
# === STEP 5: VISUALIZATION (ADAPTED FOR REAL DATA) ============================
# ==============================================================================

def visualize_attack_graph(driver):
    """Fetches the graph and highlights nodes involved in attacks."""
    print("\n--- Generating Graph Visualization ---")
    query = """
    MATCH (a:IP)-[r:CONNECTS]->(v:IP) 
    RETURN a.address AS source, v.address AS target, r.is_attack_early AS early, r.is_attack_late AS late
    """
    with driver.session() as session:
        results = session.run(query).data()
        
    if not results:
        print("No data to visualize."); return

    G = nx.DiGraph()
    attack_edges = []
    attacker_nodes = set()
    victim_nodes = set()
    
    for record in results:
        G.add_edge(record['source'], record['target'])
        if record['early'] or record['late']:
            attack_edges.append((record['source'], record['target']))
            attacker_nodes.add(record['source'])
            victim_nodes.add(record['target'])
            
    # Define node colors based on their role in attacks
    node_colors = []
    for node in G.nodes():
        if node in attacker_nodes:
            node_colors.append('red') # Attacker
        elif node in victim_nodes:
            node_colors.append('orange') # Victim
        else:
            node_colors.append('skyblue') # Normal

    edge_colors = ['red' if edge in attack_edges else 'lightgray' for edge in G.edges()]
    
    plt.figure(figsize=(20, 16))
    pos = nx.spring_layout(G, k=0.6, iterations=50)
    nx.draw(G, pos, with_labels=True, node_color=node_colors, edge_color=edge_colors,
            node_size=2000, font_size=9, font_weight='bold', width=1.0, arrows=True)
    plt.title("Real Zeek Data Network Graph with Attack Paths Highlighted", size=20)
    plt.savefig("real_attack_graph.png")
    print("Visualization saved to real_attack_graph.png")


# ==============================================================================
# === MAIN EXECUTION BLOCK =====================================================
# ==============================================================================

if __name__ == "__main__":
    driver = None
    try:
        # Step 1: Load real data instead of generating it
        zeek_df = load_real_zeek_data()
        
        if zeek_df is not None:
            # Step 2: Connect to Neo4j and build the graph
            driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
            driver.verify_connectivity()
            build_graph_from_dataframe(driver, zeek_df)
            
            # Step 3: Run the existing analysis pipeline
            test_thesis_hypothesis(driver)
            run_verification_query(driver)
            visualize_attack_graph(driver)
            
    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        if driver:
            driver.close()
            print("\nNeo4j driver closed.")


Starting data collection from UWF dataset...
  Processing directory: 2024-02-25%20-%202024-03-03
  Processing directory: 2024-03-03%20-%202024-03-10
  Processing directory: 2024-03-10%20-%202024-03-17
  Processing directory: 2024-03-17%20-%202024-03-24
  Processing directory: 2024-03-24%20-%202024-03-31
  Processing directory: 2024-10-27%20-%202024-11-03
  Processing directory: 2024-11-03%20-%202024-11-10

Combining and cleaning data...
Data loading complete. Total cleaned rows: 1898613

Building graph from 1898613 log entries...
Database cleared. Starting fresh graph build.
Creating 357 unique IP nodes...
Creating relationships and applying ATT&CK tags...


[#DD3E]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('localhost', 7687)) (ResolvedIPv6Address(('::1', 7687, 0, 0))): OSError('No data')


An error occurred: Failed to read from defunct connection IPv4Address(('localhost', 7687)) (ResolvedIPv6Address(('::1', 7687, 0, 0)))

Neo4j driver closed.


In [13]:
combined_df

,community_id,conn_state,duration,history,src_ip_zeek,src_port_zeek,dest_ip_zeek,dest_port_zeek,local_orig,local_resp,...,resp_ip_bytes,resp_pkts,service,ts,uid,datetime,label_tactic,label_technique,label_binary,label_cve
0,1:rIv0alwySftXGaqUd9xYXvhE+Gw=,S0,0.000288,S,143.88.15.10,41841,143.88.1.19,445,False,False,...,0,0,None,1.709263e+09,CsiaSX1pkBGBkxFSj,2024-03-01 03:13:53.834000,Defense Evasion,T1078,True,none
1,1:rIv0alwySftXGaqUd9xYXvhE+Gw=,S0,0.000288,S,143.88.15.10,41841,143.88.1.19,445,False,False,...,0,0,None,1.709263e+09,CsiaSX1pkBGBkxFSj,2024-03-01 03:13:53.834000,Initial Access,Duplicate,Duplicate,Duplicate
2,1:rIv0alwySftXGaqUd9xYXvhE+Gw=,S0,0.000288,S,143.88.15.10,41841,143.88.1.19,445,False,False,...,0,0,None,1.709263e+09,CsiaSX1pkBGBkxFSj,2024-03-01 03:13:53.834000,Persistence,Duplicate,Duplicate,Duplicate
3,1:rIv0alwySftXGaqUd9xYXvhE+Gw=,S0,0.000288,S,143.88.15.10,41841,143.88.1.19,445,False,False,...,0,0,None,1.709263e+09,CsiaSX1pkBGBkxFSj,2024-03-01 03:13:53.834000,Privilege Escalation,Duplicate,Duplicate,Duplicate
4,1:asFsshVIWPAlgeiBnM2QBKvPMIU=,S0,0.000954,S,143.88.15.10,46223,143.88.1.19,445,False,False,...,0,0,None,1.709391e+09,CLqHLYkLdz4jhl6v4,2024-03-02 14:46:31.076000,Defense Evasion,T1078,True,none
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1916752,1:StTKwZCGgavKHkOQNmVnsXujVMw=,SF,0.001108,Dd,143.88.0.18,3359,143.88.0.17,53,False,False,...,86,1,dns,1.730610e+09,CJadVqkDBLfk3QkN1,2024-11-03 04:59:42.900645,none,none,False,none
1916753,1:OxFc0kcZO4mSH/YsOyK9+jC8HeY=,SF,0.000665,Dd,143.88.0.18,46550,143.88.0.17,53,False,False,...,71,1,dns,1.730610e+09,ChNbMn10Y56Yzn78x2,2024-11-03 04:59:42.914804,none,none,False,none
1916754,1:9aVKjGggBxZB8uoZUZKwmUZ0XIM=,S0,3.037969,S,143.88.11.11,48878,143.88.255.10,53,False,False,...,0,0,None,1.730610e+09,Co8Ymj2C09k2jpN8mh,2024-11-03 04:59:48.970879,none,none,False,none
1916755,1:l2ZTBUf9S2vBP0i8HCgwJ2x8ibI=,S0,3.040199,S,143.88.13.13,49326,143.88.255.10,53,False,False,...,0,0,None,1.730610e+09,C6MEjD4qQ6uoBSiVwa,2024-11-03 04:59:51.184341,none,none,False,none


In [12]:
def prepare_data_for_import():
    """Loads, cleans, and saves the Zeek data to a CSV file."""
    print("--- Step 1: Loading and Preparing Data ---")
    
    # Load the data using the function from the previous script
    zeek_df = load_real_zeek_data() 
    if zeek_df is None:
        print("Data loading failed. Aborting.")
        return None

    print("\nApplying attack mappings...")
    # Apply the TTP mapping logic to create the new columns
    attack_props = zeek_df.apply(map_to_attack_ttp, axis=1, result_type='expand')
    
    # Join the new properties back to the main dataframe
    prepared_df = zeek_df.join(attack_props)
    
    # Define the output file name
    output_filename = "zeek_import_data.csv"
    
    # IMPORTANT: Define the path to your Neo4j import directory
    # This path will vary depending on your OS and installation
    # Example for Docker: '/var/lib/neo4j/import'
    # Example for Windows Desktop: 'C:/Users/YourUser/AppData/Local/Neo4j/Relate/Data/dbmss/dbms-your-db-id/import'
    neo4j_import_path = "/path/to/your/neo4j/import" # <-- CHANGE THIS
    
    output_filepath = os.path.join(neo4j_import_path, output_filename)

    print(f"\nSaving prepared data to: {output_filepath}")
    
    # Select only the columns we need for the graph to keep the CSV lean
    columns_to_export = [
        'src_ip_zeek', 'dest_ip_zeek', 'ts', 'duration', 'service', 
        'dest_port_zeek', 'conn_state', 'tactic', 'ttp_tag', 
        'is_attack', 'is_attack_early', 'is_attack_late'
    ]
    prepared_df[columns_to_export].to_csv(output_filepath, index=False)
    
    print("--- Data preparation complete! ---")
    return output_filename

def build_graph_with_load_csv(driver, filename):
    """Builds the graph by executing a LOAD CSV Cypher query."""
    print("\n--- Step 2: Building Graph with LOAD CSV ---")
    
    with driver.session() as session:
        # Clear the database
        session.run("MATCH (n) DETACH DELETE n")
        print("Database cleared.")

        # This is the powerful LOAD CSV query
        # It uses PERIODIC COMMIT to handle large files without running out of memory
        load_query = """
        USING PERIODIC COMMIT 50000
        LOAD CSV WITH HEADERS FROM 'file:///{filename}' AS row
        MERGE (orig:IP {{address: row.src_ip_zeek}})
        MERGE (resp:IP {{address: row.dest_ip_zeek}})
        CREATE (orig)-[:CONNECTS {{
            timestamp: toFloat(row.ts), 
            duration: toFloat(row.duration), 
            service: row.service,
            port: toInteger(row.dest_port_zeek), 
            state: row.conn_state, 
            tactic: row.tactic,
            ttp_tag: row.ttp_tag, 
            is_attack: toBoolean(row.is_attack), 
            is_attack_early: toBoolean(row.is_attack_early), 
            is_attack_late: toBoolean(row.is_attack_late)
        }}]->(resp)
        """.format(filename=filename)

        print("Executing LOAD CSV query. This may take a few minutes...")
        session.run(load_query)
        print("--- Graph construction complete! ---")


if __name__ == "__main__":
    driver = None
    try:
        # Step 1: Prepare the data and save it to CSV in the import folder
        import_filename = prepare_data_for_import()
        
        if import_filename:
            # Step 2: Connect to Neo4j and run the LOAD CSV command
            driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
            driver.verify_connectivity()
            build_graph_with_load_csv(driver, import_filename)
            
            # Step 3: The rest of your analysis pipeline can now run as before!
            print("\nProceeding with analysis pipeline...")
            test_thesis_hypothesis(driver)
            run_verification_query(driver)
            visualize_attack_graph(driver)
            
    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        if driver:
            driver.close()
            print("\nNeo4j driver closed.")

--- Step 1: Loading and Preparing Data ---
Starting data collection from UWF dataset...
  Processing directory: 2024-02-25%20-%202024-03-03
  Processing directory: 2024-03-03%20-%202024-03-10
  Processing directory: 2024-03-10%20-%202024-03-17
  Processing directory: 2024-03-17%20-%202024-03-24
  Processing directory: 2024-03-24%20-%202024-03-31
  Processing directory: 2024-10-27%20-%202024-11-03
  Processing directory: 2024-11-03%20-%202024-11-10

Combining and cleaning data...
Data loading complete. Total cleaned rows: 1898613

Applying attack mappings...

Saving prepared data to: /path/to/your/neo4j/import/zeek_import_data.csv
An error occurred: "['is_attack_early', 'is_attack_late'] not in index"
